In [1]:
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import ussa1976

# Pick an Altitude

alt = np.array([90000]) # m

# Get properties of the atmosphere at that altitude
ds = ussa1976.compute(z=alt)
rho = ds['rho'][0]
n_tot = ds['n_tot'][0]
t = ds['t'][0]
p = ds['p'][0]
mfp = ds['mfp'][0]
f = ds['f'][0]
v = ds['v'][0]

print(f"At {alt[0]/1000:.1f} km, the atmosphere has:")
print(f"  - Density: {rho:.2e} kg/m³")
print(f"  - Number density: {n_tot:.2e} m⁻³")
print(f"  - Temperature: {t:.2f} K")
print(f"  - Pressure: {p:.2f} Pa")
print(f"  - Mean free path: {mfp:.2e} m")
print(f"  - Collision frequency: {f:.2e} Hz")
print(f"  - Average molecular speed: {v:.2f} m/s\n")


# Domain Information
x_min = -1e-1  # m
x_max = 1e-1   # m
y_min = -1e-1  # m
y_max = 1e-1   # m
lx = x_max - x_min
ly = y_max - y_min
print(f"Domain size: {lx:.2e} m x {ly:.2e} m")

grid_size_mfp_ratio = 0.25
grid_size = mfp * grid_size_mfp_ratio
nx = int(lx / grid_size)
ny = int(ly / grid_size)

print(f"Grid size: {grid_size:.2e} m")
print(f"Number of grid points: {nx} x {ny} = {nx*ny}") 

target_particles_per_cell = 100
num_real_particles = n_tot * lx * ly
num_sim_particles = int(target_particles_per_cell * nx * ny)
weight = num_real_particles / num_sim_particles
print(f"Number of real particles in domain: {num_real_particles:.2e}")
print(f"Number of simulated particles: {num_sim_particles:.2e}")
print(f"Weight (real particles per simulated particle): {weight:.2e}\n")

timestep_to_mct_ratio = 0.25
timestep = 1 / f * timestep_to_mct_ratio
print(f"Time step: {timestep:.2e} s")

inflow_velocity = 5000 # m/s
dist_moved_per_timestep = inflow_velocity * timestep
dist_moved_per_timestep_to_grid_size_ratio = dist_moved_per_timestep / grid_size
print(f"Distance moved per time step: {dist_moved_per_timestep:.2e} m")
print(f"Distance moved per time step to grid size ratio: {dist_moved_per_timestep_to_grid_size_ratio:.2f}")

timestep_that_particles_move_one_grid_cell = grid_size / inflow_velocity
print(f"Time step that particles move one grid cell: {timestep_that_particles_move_one_grid_cell:.2e} s")

largest_dimension = max(lx, ly)
time_to_cross_domain = largest_dimension / v
print(f"Physical time until steady state: {time_to_cross_domain:.2e} s")

most_restrictive_time_step = min(timestep, timestep_that_particles_move_one_grid_cell)
if timestep < timestep_that_particles_move_one_grid_cell:
    print("Most restrictive time step is based on collision frequency.")
    print(f"Time step based on collision frequency: {timestep:.2e} s")
else:
    print("Most restrictive time step is based on particle movement across grid cells.")
    print(f"Time step based on particle movement: {timestep_that_particles_move_one_grid_cell:.2e} s")

num_time_steps = int(time_to_cross_domain / most_restrictive_time_step)
print(f"Number of time steps to reach steady state: {num_time_steps:.2e}\n")

speed_ratio = inflow_velocity / np.sqrt(2 * 287 * t)
print(f"Speed ratio (inflow velocity / thermal speed): {speed_ratio:.2f}")

Kn = 2e-2 / mfp
print(f"Knudsen number (characteristic length / mean free path): {Kn:.2e}")

At 90.0 km, the atmosphere has:
  - Density: 3.42e-06 kg/m³
  - Number density: 7.12e+19 m⁻³
  - Temperature: 186.87 K
  - Pressure: 0.18 Pa
  - Mean free path: 2.37e-02 m
  - Collision frequency: 1.56e+04 Hz
  - Average molecular speed: 369.94 m/s

Domain size: 2.00e-01 m x 2.00e-01 m
Grid size: 5.93e-03 m
Number of grid points: 33 x 33 = 1089
Number of real particles in domain: 2.85e+18
Number of simulated particles: 1.09e+05
Weight (real particles per simulated particle): 2.61e+13

Time step: 1.60e-05 s
Distance moved per time step: 8.02e-02 m
Distance moved per time step to grid size ratio: 13.52
Time step that particles move one grid cell: 1.19e-06 s
Physical time until steady state: 5.41e-04 s
Most restrictive time step is based on particle movement across grid cells.
Time step based on particle movement: 1.19e-06 s
Number of time steps to reach steady state: 4.55e+02

Speed ratio (inflow velocity / thermal speed): 15.27
Knudsen number (characteristic length / mean free path): 8.